# 01 Dataset Inspection and Cleaning
 
 Discover the MalDroid/CICMalDroid label files and JSON behavior reports, build a stable ordered master dataframe, immediately split it into two contiguous raw shards, and then inspect/clean only the assigned shard.


1-This setup cell installs the Python packages used by this notebook inside the active notebook kernel.


In [1]:
# CHANGED: install project dependencies directly inside notebook 01 instead of using a requirements file.
%pip install -q pandas ipython jupyter ipykernel tqdm python-dotenv requests groq


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2-This cell imports the core libraries and defines the shard assignment, deletion behavior, and project paths used by the rest of the notebook.

In [2]:
from pathlib import Path
import json
import re
from collections import Counter
from functools import reduce

import pandas as pd
from IPython.display import display

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, total=None):
        return iterable

# Each machine edits this value and keeps only its assigned intermediate raw shard.
ASSIGNED_SHARD_ID = 2  # 1 or 2
DELETE_UNASSIGNED_SHARD = False
NUM_SHARDS = 2

# Make paths work from either the project root or the notebooks/ folder.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
# CHANGED: raw MalDroid 2020 dataset now lives outside the repo on E:\dataset.
RAW_DIR = Path(r"E:\dataset")
CAPTURING_LOGS_DIR = RAW_DIR / "Capturing-logs"
JSON_DIR = CAPTURING_LOGS_DIR / "Data_Merged"
CSV_DIR = RAW_DIR / "CSV"
# CHANGED: generated/intermediate outputs stay on E: to avoid filling the C: drive.
OUTPUT_ROOT = Path(r"E:\PFE_CoT_outputs")
INTERIM_DIR = OUTPUT_ROOT / "data" / "interim"

RAW_SHARD_PATHS = {
    1: INTERIM_DIR / "raw_shard1.csv",
    2: INTERIM_DIR / "raw_shard2.csv",
}

SOURCE_DATASET = "CICMalDroid2020"
JSON_SUMMARY_TOP_N = 20
MAX_DYNAMIC_EVENTS_TO_SCAN = 5000  # Caps per-sample event scanning , tried 1000 , now 3000 , now 5000
REBUILD_JSON_SUMMARY = False
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

if ASSIGNED_SHARD_ID not in RAW_SHARD_PATHS:
    raise ValueError("ASSIGNED_SHARD_ID must be 1 or 2.")

ASSIGNED_RAW_SHARD_PATH = RAW_SHARD_PATHS[ASSIGNED_SHARD_ID]
CLEANED_PATH = INTERIM_DIR / f"cleaned_cicmaldroid_shard{ASSIGNED_SHARD_ID}.csv"
JSON_SUMMARY_PATH = INTERIM_DIR / f"json_behavior_summary_shard{ASSIGNED_SHARD_ID}.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data folder:", RAW_DIR)
print("JSON behavior folder:", JSON_DIR)
print("CSV folder:", CSV_DIR)
print("Assigned shard:", ASSIGNED_SHARD_ID)
print("Delete unassigned shard:", DELETE_UNASSIGNED_SHARD)
print("Assigned raw shard:", ASSIGNED_RAW_SHARD_PATH)
print("Assigned cleaned output:", CLEANED_PATH)
print("Assigned JSON summary cache:", JSON_SUMMARY_PATH)

# CHANGED: RAW_DIR can be outside PROJECT_ROOT, so display absolute paths when needed.
def display_path(path):
    try:
        return str(path.relative_to(PROJECT_ROOT))
    except ValueError:
        return str(path)

Project root: C:\Users\milkw\Desktop\PFE_exp\CoT_generation
Raw data folder: E:\dataset
JSON behavior folder: E:\dataset\Capturing-logs\Data_Merged
CSV folder: E:\dataset\CSV
Assigned shard: 2
Delete unassigned shard: False
Assigned raw shard: E:\PFE_CoT_outputs\data\interim\raw_shard2.csv
Assigned cleaned output: E:\PFE_CoT_outputs\data\interim\cleaned_cicmaldroid_shard2.csv
Assigned JSON summary cache: E:\PFE_CoT_outputs\data\interim\json_behavior_summary_shard2.csv


3-This cell discovers the MalDroid/CICMalDroid raw files, including label text files, JSON behavior reports, and CSV feature files.

In [3]:
# Raw discovery is allowed before splitting. Heavy JSON parsing happens only after the assigned shard is selected.
raw_files = sorted(path for path in RAW_DIR.rglob("*") if path.is_file())
if not raw_files:
    raise FileNotFoundError(f"No files found in {RAW_DIR}. Place the MalDroid/CICMalDroid files there first.")

file_inventory = pd.DataFrame([
    {
        "path": display_path(path),
        "suffix": path.suffix.lower(),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
    }
    for path in raw_files
])
display(file_inventory.head(100))
print("Total raw files discovered:", len(file_inventory))

label_paths = sorted(path for path in CAPTURING_LOGS_DIR.glob("*.txt") if path.is_file())
json_paths = sorted(path for path in JSON_DIR.glob("*.json") if path.is_file())
csv_paths = sorted(path for path in CSV_DIR.rglob("*.csv") if path.is_file()) if CSV_DIR.exists() else []

if not label_paths:
    raise FileNotFoundError(f"No label text files found in {CAPTURING_LOGS_DIR}.")
if not json_paths:
    raise FileNotFoundError(f"No JSON behavior files found in {JSON_DIR}.")

print("Label files selected:")
for path in label_paths:
    print("-", display_path(path))

print("\nJSON behavior files discovered:", len(json_paths))
print("First JSON files:")
for path in json_paths[:10]:
    print("-", display_path(path))

print("\nCSV feature files discovered:", len(csv_paths))
for path in csv_paths:
    print("-", display_path(path))

,path,suffix,size_mb
0,E:\dataset\Capturing-logs\adware.txt,.txt,0.102
1,E:\dataset\Capturing-logs\banking.txt,.txt,0.168
2,E:\dataset\Capturing-logs\benign.txt,.txt,0.158
3,E:\dataset\Capturing-logs\Data_Merged\00049D03...,.json,11.029
4,E:\dataset\Capturing-logs\Data_Merged\00050EE1...,.json,2.118
...,...,...,...
95,E:\dataset\Capturing-logs\Data_Merged\01AE30AE...,.json,2.467
96,E:\dataset\Capturing-logs\Data_Merged\01AFCF5F...,.json,77.215
97,E:\dataset\Capturing-logs\Data_Merged\01AFDB81...,.json,33.516
98,E:\dataset\Capturing-logs\Data_Merged\01BA8339...,.json,38.559


Total raw files discovered: 13309
Label files selected:
- E:\dataset\Capturing-logs\adware.txt
- E:\dataset\Capturing-logs\banking.txt
- E:\dataset\Capturing-logs\benign.txt
- E:\dataset\Capturing-logs\riskware.txt
- E:\dataset\Capturing-logs\sms.txt

JSON behavior files discovered: 13295
First JSON files:
- E:\dataset\Capturing-logs\Data_Merged\00049D038A2ABC2D5FE3B190D6CF5C1CB1BA63441DEFDF136BE251C7A00727D8.json
- E:\dataset\Capturing-logs\Data_Merged\00050EE1EFEE0602426671153AD0487A46C235D5FFB397427C96C832A6DB38A1.json
- E:\dataset\Capturing-logs\Data_Merged\0008B849C70477FAF464E80DF6C7EA91569F78BD2F13AF79924F8B5341A44EDD.json
- E:\dataset\Capturing-logs\Data_Merged\000AEC00A2B667AAF1224974ABA7EE30B4B088E073368C605EE335FB8EE11B54.json
- E:\dataset\Capturing-logs\Data_Merged\000E0948176BDEC2B6E19D0F03E23F37910676A9B7E7709954614BAC79269C36.json
- E:\dataset\Capturing-logs\Data_Merged\000E7149AB7550EF605C2B22CB1BEAFFBEE9219699661D89158D490A3FFA393A.json
- E:\dataset\Capturing-logs\Data

4-This cell defines helper functions for stable identifier extraction, label loading, CSV inspection, and compact JSON behavior summarization.

In [4]:
def load_csv(path):
    try:
        frame = pd.read_csv(path, low_memory=False)
    except UnicodeDecodeError:
        frame = pd.read_csv(path, low_memory=False, encoding="latin-1")




def clean_column_names(columns):
    cleaned = []
    seen = {}
    for column in columns:
        base = str(column).strip()
        count = seen.get(base, 0)
        cleaned.append(base if count == 0 else f"{base}__duplicate_{count}")
        seen[base] = count + 1
    return cleaned


def normalize_name(name):
    return re.sub(r"[^a-z0-9]+", "", str(name).lower())


def find_first_column(columns, candidates):
    normalized = {normalize_name(column): column for column in columns}
    for candidate in candidates:
        key = normalize_name(candidate)
        if key in normalized:
            return normalized[key]
    for column in columns:
        col_key = normalize_name(column)
        if any(normalize_name(candidate) in col_key for candidate in candidates):
            return column
    return None


def clean_identifier_value(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except TypeError:
        pass
    text = str(value).strip()
    return text


def extract_sha256_from_name(value):
    match = re.search(r"([A-Fa-f0-9]{64})", str(value))
    if match:
        return match.group(1).lower()
    return Path(str(value)).stem.lower()


def category_from_label_file(path):
    mapping = {
        "adware": "Adware",
        "banking": "Banking",
        "benign": "Benign",
        "riskware": "Riskware",
        "sms": "SMS",
    }
    return mapping.get(path.stem.lower(), path.stem)


def compact_list(values, limit=JSON_SUMMARY_TOP_N):
    if not isinstance(values, list):
        return "unknown"
    cleaned = [str(value).strip() for value in values if str(value).strip()]
    if not cleaned:
        return "unknown"
    return "; ".join(cleaned[:limit])


def compact_counter(counter, limit=JSON_SUMMARY_TOP_N):
    if not counter:
        return "unknown"
    return "; ".join(f"{name}={count}" for name, count in counter.most_common(limit))


def compact_nonzero_dict(values, limit=JSON_SUMMARY_TOP_N):
    if not isinstance(values, dict):
        return "unknown"
    nonzero = {}
    for key, value in values.items():
        try:
            numeric_value = float(value)
        except (TypeError, ValueError):
            numeric_value = 1 if value else 0
        if numeric_value != 0:
            nonzero[str(key)] = int(numeric_value) if numeric_value.is_integer() else numeric_value
    if not nonzero:
        return "none"
    ordered = sorted(nonzero.items(), key=lambda item: (-float(item[1]), item[0]))[:limit]
    return "; ".join(f"{key}={value}" for key, value in ordered)


def event_action_name(low_event):
    for key in ["sysname", "method", "api", "name", "operation", "action"]:
        value = low_event.get(key) if isinstance(low_event, dict) else None
        if value:
            return str(value)
    return None


def summarize_dynamic_host(host_events):
    event_class_counts = Counter()
    low_type_counts = Counter()
    action_counts = Counter()
    host_event_count = len(host_events) if isinstance(host_events, list) else 0
    scanned_events = 0
    scanned_low_events = 0

    if not isinstance(host_events, list):
        return host_event_count, scanned_events, scanned_low_events, event_class_counts, low_type_counts, action_counts

    for event in host_events:
        if MAX_DYNAMIC_EVENTS_TO_SCAN is not None and scanned_events >= MAX_DYNAMIC_EVENTS_TO_SCAN:
            break
        scanned_events += 1
        if not isinstance(event, dict):
            continue

        event_class = str(event.get("class") or "unknown")
        event_class_counts[event_class] += 1

        low_events = event.get("low", [])
        if isinstance(low_events, dict):
            low_events = [low_events]
        if not isinstance(low_events, list):
            low_events = []

        for low_event in low_events:
            if not isinstance(low_event, dict):
                continue
            scanned_low_events += 1
            low_type_counts[str(low_event.get("type") or event_class)] += 1
            action_name = event_action_name(low_event)
            if action_name:
                action_counts[action_name] += 1

    return host_event_count, scanned_events, scanned_low_events, event_class_counts, low_type_counts, action_counts


def summarize_json_behavior(json_path):
    summary = {
        "json_loaded": False,
        "json_parse_error": "",
        "json_sha256": extract_sha256_from_name(json_path.name),
        "json_md5": "unknown",
        "json_process_name": "unknown",
        "json_package": "unknown",
        "static_num_permissions": "unknown",
        "static_permissions": "unknown",
        "static_intent_actions": "unknown",
        "static_receivers": "unknown",
        "static_services": "unknown",
        "static_sensitive_api_nonzero": "unknown",
        "dynamic_host_event_count": 0,
        "dynamic_scanned_event_count": 0,
        "dynamic_scanned_low_event_count": 0,
        "dynamic_event_classes": "unknown",
        "dynamic_low_types": "unknown",
        "dynamic_top_actions": "unknown",
        "json_behavior_summary": "unknown",
    }

    try:
        with open(json_path, "r", encoding="utf-8", errors="replace") as input_handle:
            data = json.load(input_handle)
    except Exception as error:
        summary["json_parse_error"] = f"{type(error).__name__}: {error}"
        summary["json_behavior_summary"] = f"json_parse_error: {summary['json_parse_error']}"
        return summary

    if not isinstance(data, dict):
        summary["json_parse_error"] = f"Expected dict JSON root, got {type(data).__name__}"
        summary["json_behavior_summary"] = summary["json_parse_error"]
        return summary

    behaviors = data.get("behaviors", {}) if isinstance(data.get("behaviors", {}), dict) else {}
    static = behaviors.get("static", {}) if isinstance(behaviors.get("static", {}), dict) else {}
    dynamic = behaviors.get("dynamic", {}) if isinstance(behaviors.get("dynamic", {}), dict) else {}

    permissions = static.get("permissions")
    if not isinstance(permissions, list) and isinstance(static.get("used_permissions"), dict):
        permissions = list(static["used_permissions"].keys())

    host_events = dynamic.get("host", [])
    (
        host_event_count,
        scanned_events,
        scanned_low_events,
        event_class_counts,
        low_type_counts,
        action_counts,
    ) = summarize_dynamic_host(host_events)

    summary.update({
        "json_loaded": True,
        "json_sha256": clean_identifier_value(data.get("sha256")) or extract_sha256_from_name(json_path.name),
        "json_md5": clean_identifier_value(data.get("md5")) or "unknown",
        "json_process_name": clean_identifier_value(data.get("process_name")) or "unknown",
        "json_package": clean_identifier_value(static.get("package")) or "unknown",
        "static_num_permissions": static.get("num_permissions", len(permissions) if isinstance(permissions, list) else "unknown"),
        "static_permissions": compact_list(permissions),
        "static_intent_actions": compact_list(static.get("intent_actions")),
        "static_receivers": compact_list(static.get("receivers")),
        "static_services": compact_list(static.get("services")),
        "static_sensitive_api_nonzero": compact_nonzero_dict(static.get("sensitive_API")),
        "dynamic_host_event_count": host_event_count,
        "dynamic_scanned_event_count": scanned_events,
        "dynamic_scanned_low_event_count": scanned_low_events,
        "dynamic_event_classes": compact_counter(event_class_counts),
        "dynamic_low_types": compact_counter(low_type_counts),
        "dynamic_top_actions": compact_counter(action_counts),
    })

    truncated_note = ""
    if MAX_DYNAMIC_EVENTS_TO_SCAN is not None and host_event_count > scanned_events:
        truncated_note = f"; dynamic_events_truncated_after={scanned_events}"

    summary["json_behavior_summary"] = "\n".join([
        f"static_num_permissions: {summary['static_num_permissions']}",
        f"static_permissions: {summary['static_permissions']}",
        f"static_intent_actions: {summary['static_intent_actions']}",
        f"static_sensitive_api_nonzero: {summary['static_sensitive_api_nonzero']}",
        f"dynamic_host_event_count: {host_event_count}; dynamic_scanned_event_count: {scanned_events}; dynamic_scanned_low_event_count: {scanned_low_events}{truncated_note}",
        f"dynamic_event_classes: {summary['dynamic_event_classes']}",
        f"dynamic_low_types: {summary['dynamic_low_types']}",
        f"dynamic_top_actions: {summary['dynamic_top_actions']}",
    ])
    return summary


IDENTIFIER_CANDIDATES = [
    "original_id", "dataset_key", "key", "sample_key", "json_filename", "json_file",
    "sha256", "sha1", "md5", "hash", "apk_hash", "apk", "apk_name", "apk_filename",
    "file", "filename", "file_name", "path", "file_path", "app", "app_name",
    "package", "package_name", "pkg_name", "name", "raw_identifier",
]
ORIGINAL_ID_CANDIDATES = [
    "original_id", "dataset_key", "key", "sample_key", "json_filename", "json_file",
    "apk_name", "apk_filename", "apk", "filename", "file_name", "file_path", "path",
    "sha256", "sha1", "md5", "hash", "apk_hash", "raw_identifier", "package_name",
]
ORDER_CANDIDATES = [
    "filename", "file_name", "file_path", "path", "apk_name", "apk", "file", "name",
]
LABEL_CANDIDATES = [
    "true_label", "label", "class", "classification", "malware", "is_malware",
    "benign_malware", "type", "verdict",
]
CATEGORY_CANDIDATES = [
    "category", "malware_category", "malware_type", "class", "classification", "type",
]
FAMILY_CANDIDATES = [
    "family", "malware_family", "family_name", "malware_name", "variant", "subclass",
]

5-This cell deduplicates and filters JSON filenames using only lightweight filename and label metadata before any JSON summaries are built.

In [5]:
# Deduplicate and filter raw JSON filenames before building master_df or parsing JSON contents.
raw_json_paths = sorted(path for path in JSON_DIR.glob("*.json") if path.is_file())
original_json_file_count = len(raw_json_paths)

labels_by_hash = {}
label_sources_by_hash = {}
for label_path in label_paths:
    category = category_from_label_file(label_path)
    with open(label_path, "r", encoding="utf-8", errors="replace") as input_handle:
        for line in input_handle:
            raw_name = line.strip()
            if not raw_name:
                continue
            file_hash = extract_sha256_from_name(raw_name)
            labels_by_hash.setdefault(file_hash, []).append(category)
            label_sources_by_hash.setdefault(file_hash, []).append(label_path.name)

seen_json_hashes = set()
deduped_json_records = []
duplicate_json_hashes = []
for path in raw_json_paths:
    file_hash = extract_sha256_from_name(path.name)
    if file_hash in seen_json_hashes:
        duplicate_json_hashes.append(file_hash)
        continue
    seen_json_hashes.add(file_hash)
    deduped_json_records.append((path, file_hash))

print("Duplicate JSON files by SHA256 dropped:", len(duplicate_json_hashes))
if duplicate_json_hashes:
    print("First duplicate hashes:", sorted(set(duplicate_json_hashes))[:10])

labeled_json_records = []
unlabeled_json_hashes = []
for path, file_hash in deduped_json_records:
    if file_hash not in labels_by_hash:
        unlabeled_json_hashes.append(file_hash)
        continue
    labeled_json_records.append((path, file_hash))

print("JSON files without a matching label entry dropped:", len(unlabeled_json_hashes))

conflicting_label_hashes = sorted(
    file_hash for file_hash, categories in labels_by_hash.items()
    if len(set(categories)) > 1
)
conflicting_label_hash_set = set(conflicting_label_hashes)
conflicting_json_hashes = [
    file_hash for _, file_hash in labeled_json_records
    if file_hash in conflicting_label_hash_set
]

print("Ambiguous label hashes dropped:", len(conflicting_label_hashes))
if conflicting_label_hashes:
    print("First conflicting label hashes:", conflicting_label_hashes[:10])
print("JSON files dropped because labels were ambiguous:", len(conflicting_json_hashes))

json_paths = [
    path for path, file_hash in labeled_json_records
    if file_hash not in conflicting_label_hash_set
]
retained_json_hashes = {extract_sha256_from_name(path.name) for path in json_paths}
cleaned_labels_by_hash = {
    file_hash: sorted(set(categories))
    for file_hash, categories in labels_by_hash.items()
    if file_hash in retained_json_hashes and file_hash not in conflicting_label_hash_set
}
cleaned_label_sources_by_hash = {
    file_hash: sorted(set(label_sources_by_hash.get(file_hash, [])))
    for file_hash in cleaned_labels_by_hash
}
labels_by_hash = cleaned_labels_by_hash
label_sources_by_hash = cleaned_label_sources_by_hash

dropped_json_file_count = original_json_file_count - len(json_paths)
cleaned_category_counts = Counter(categories[0] for categories in labels_by_hash.values() if categories)

print("\nFilename/label metadata cleanup summary")
print("Original JSON file count:", original_json_file_count)
print("Dropped JSON file count:", dropped_json_file_count)
print("Remaining JSON file count:", len(json_paths))
print("Category distribution from cleaned labels_by_hash:")
if cleaned_category_counts:
    print(pd.Series(cleaned_category_counts).sort_index())
else:
    print(pd.Series(dtype=int))


Duplicate JSON files by SHA256 dropped: 45
First duplicate hashes: ['05b33442670e460c893710b7c0dda46bde826d8067bbaba36d1ee0d5907207ac', '07f9a055fdf9e3e67bfe7a67952747c0020e3e4ffe461122d23b653d4fd52455', '08b82c25047da6fa928c17ae91e04c8fdf6f258919381ead6ec1ee9605c0bf6c', '124860e81b32724dc6c050dfa9d736e779e8ff4bf404cef2a8423b57c4a695f5', '144debead52cc5deabedf91cf6379b20189b87938b0e4b1cb28e49e7c4ed9c16', '182b9e93ffefc2c4de1b14fff9ac1310ffcaf2c578bb3acd17328d1b88dacbfb', '191b4eb236c5ef2dfe5b942262d01d118ebf5c9a225ef7f0cba5a184445783aa', '19b9de0d94fd6f9508bb1a02bbd3f6d05f304e0e14f278b833b8be2a7352e609', '3386a5a5ee447cbde467e26f8442bcd2f9ada8eda03f8ca2e46e39b19aa4debb', '36e15c8b6211b22d4176424339ab39a52e65d2b1c9dea3b24c3639fb022a85ec']
JSON files without a matching label entry dropped: 604
Ambiguous label hashes dropped: 2
First conflicting label hashes: ['00847524ec1e69b2cdd53205cd9725295e87094eeed1567b3efb12191ded24d2', '97bfb35785c155686de96095e3c87e243e0fdcd37976e7c7254e48f66c75d

6-This cell builds a JSON-keyed master dataframe from the cleaned label map and deduplicated behavior JSON filenames, without loading the huge JSON contents yet.

In [6]:
# Build one stable row per already-clean JSON behavior report.
master_rows = []
for row_order, path in enumerate(json_paths):
    file_hash = extract_sha256_from_name(path.name)
    categories = labels_by_hash.get(file_hash, [])
    unique_categories = sorted(set(categories))
    category = unique_categories[0] if unique_categories else "unknown"
    master_rows.append({
        "raw_identifier": file_hash,
        "json_file": path.name,
        "json_path": str(path),
        "sha256": file_hash,
        "true_label": category,
        "category": category,
        "family": "unknown",
        "label_source_file": "; ".join(sorted(set(label_sources_by_hash.get(file_hash, [])))) or "unknown",
        "label_conflict": len(unique_categories) > 1,
        "source_file": display_path(path),
        "source_row_order": row_order,
    })

master_df = pd.DataFrame(master_rows)
json_hashes = set(master_df["raw_identifier"])
label_hashes = set(labels_by_hash)
missing_json_for_labels = sorted(label_hashes - json_hashes)
unlabeled_json_hashes = sorted(json_hashes - label_hashes)
conflicting_label_hashes = sorted(
    file_hash for file_hash, categories in labels_by_hash.items()
    if len(set(categories)) > 1
)

print("JSON behavior files loaded into manifest:", len(master_df))
print("Unique JSON hashes:", master_df["raw_identifier"].nunique())
print("Cleaned label hashes available:", len(label_hashes))
print("Labeled hashes missing JSON files:", len(missing_json_for_labels))
print("JSON files without label text entry:", len(unlabeled_json_hashes))
print("Hashes with conflicting category labels:", len(conflicting_label_hashes))
print("Category distribution from label files:")
print(master_df["category"].value_counts(dropna=False))

if missing_json_for_labels:
    print("First missing labeled hashes:", missing_json_for_labels[:10])
if unlabeled_json_hashes:
    print("First unlabeled JSON hashes:", unlabeled_json_hashes[:10])
if conflicting_label_hashes:
    print("First conflicting label hashes:", conflicting_label_hashes[:10])

# The CSV feature matrices are kept as raw traceability inputs. They do not expose a stable hash/file key,
# so this clean CoT stage avoids row-order joins and uses the JSON behavior reports instead.
csv_header_summary = []
for path in csv_paths:
    try:
        header = pd.read_csv(path, nrows=0)
        columns = clean_column_names(header.columns)
        id_column = find_first_column(columns, IDENTIFIER_CANDIDATES)
        label_column = find_first_column(columns, LABEL_CANDIDATES)
        csv_header_summary.append({
            "csv_file": display_path(path),
            "columns": len(columns),
            "identifier_column_detected": id_column or "none",
            "label_column_detected": label_column or "none",
        })
    except Exception as error:
        csv_header_summary.append({
            "csv_file": display_path(path),
            "columns": "unknown",
            "identifier_column_detected": "read_error",
            "label_column_detected": f"{type(error).__name__}: {error}",
        })

if csv_header_summary:
    print("\nCSV header inspection:")
    display(pd.DataFrame(csv_header_summary))

JSON behavior files loaded into manifest: 12644
Unique JSON hashes: 12644
Cleaned label hashes available: 12644
Labeled hashes missing JSON files: 0
JSON files without label text entry: 0
Hashes with conflicting category labels: 0
Category distribution from label files:
category
SMS         3990
Riskware    2974
Benign      2184
Banking     2160
Adware      1336
Name: count, dtype: int64

CSV header inspection:


,csv_file,columns,identifier_column_detected,label_column_detected
0,E:\dataset\CSV\feature_vectors_static.csv,50621,APK,'+AC0ALQAt-0+AC0ALQAtAC0ALQ-viewClassName+AD0-...
1,E:\dataset\CSV\feature_vectors_syscalls_freque...,140,getpeername,Class
2,E:\dataset\CSV\feature_vectors_syscallsbinders...,471,FS_ACCESS(CREATE__APPEND)__,Class
3,E:\dataset\CSV\syscall_unique.csv,1,none,none


7-This cell builds the deterministic master ordering, creates `sample_id` and `original_id`, then writes the two contiguous raw shard CSV files.

In [7]:
# Build deterministic ordering before any cleaning or prompt generation.
order_search_columns = [
    column for column in master_df.columns
    if not normalize_name(column).endswith("sourcefile") and normalize_name(column) != "sourceroworder"
]
order_column = find_first_column(order_search_columns, ORDER_CANDIDATES)
sort_columns = []
temporary_sort_columns = []

if order_column:
    master_df["__sort_original_path"] = master_df[order_column].astype("string").fillna("")
    sort_columns.append("__sort_original_path")
    temporary_sort_columns.append("__sort_original_path")
    print("Primary ordering column:", order_column)
else:
    print("Primary ordering column: none detected; using stable source file and row order.")

if "source_file" in master_df.columns:
    master_df["__sort_source_file"] = master_df["source_file"].astype("string").fillna("")
else:
    master_df["__sort_source_file"] = ""
sort_columns.append("__sort_source_file")
temporary_sort_columns.append("__sort_source_file")

if "source_row_order" in master_df.columns:
    master_df["__sort_row_order"] = pd.to_numeric(master_df["source_row_order"], errors="coerce").fillna(0).astype(int)
else:
    master_df["__sort_row_order"] = range(len(master_df))
sort_columns.append("__sort_row_order")
temporary_sort_columns.append("__sort_row_order")

master_df = master_df.sort_values(sort_columns, kind="mergesort").drop(columns=temporary_sort_columns).reset_index(drop=True)

# Preserve any existing columns before creating the canonical pipeline columns.
if "sample_id" in master_df.columns:
    master_df = master_df.rename(columns={"sample_id": "original_sample_id"})
if "source_dataset" in master_df.columns:
    master_df = master_df.rename(columns={"source_dataset": "original_source_dataset"})
if "original_id" in master_df.columns:
    master_df = master_df.rename(columns={"original_id": "raw_original_id"})

# original_id is the final merge key. It is built before sharding and never dropped.
original_id_source_col = None
if "raw_original_id" in master_df.columns:
    original_id_source_col = "raw_original_id"
elif "raw_identifier" in master_df.columns:
    original_id_source_col = "raw_identifier"
else:
    original_id_source_col = find_first_column(order_search_columns, ORIGINAL_ID_CANDIDATES)

fallback_source = (
    master_df["source_file"].astype("string").fillna("unknown_source")
    if "source_file" in master_df.columns
    else pd.Series(["unknown_source"] * len(master_df), index=master_df.index)
)
if "source_row_order" in master_df.columns:
    row_order_fallback = pd.Series(master_df.index, index=master_df.index)
    fallback_row = pd.to_numeric(master_df["source_row_order"], errors="coerce").fillna(row_order_fallback).astype(int).astype(str)
else:
    fallback_row = pd.Series(master_df.index.astype(str), index=master_df.index)
fallback_ids = fallback_source.astype(str) + "::row_" + fallback_row.astype(str)

if original_id_source_col and original_id_source_col in master_df.columns:
    base_original_ids = master_df[original_id_source_col].map(clean_identifier_value)
    base_original_ids = base_original_ids.where(base_original_ids != "", fallback_ids)
else:
    base_original_ids = fallback_ids
    original_id_source_col = "source_file + source_row_order fallback"

duplicate_base_count = int(base_original_ids.duplicated().sum())
occurrence = base_original_ids.groupby(base_original_ids).cumcount()
original_ids = base_original_ids.where(occurrence == 0, base_original_ids + "::dup_" + occurrence.astype(str))

master_df.insert(0, "sample_id", range(len(master_df)))
master_df.insert(1, "original_id", original_ids.astype(str))
master_df.insert(2, "source_dataset", SOURCE_DATASET)

print("Original ID source:", original_id_source_col)
print("Duplicate base original IDs before deterministic suffixing:", duplicate_base_count)
print("Unique original_id values:", master_df["original_id"].nunique())

midpoint = len(master_df) // 2
raw_shards = {
    1: master_df.iloc[:midpoint].copy().reset_index(drop=True),
    2: master_df.iloc[midpoint:].copy().reset_index(drop=True),
}

for shard_id, shard_df in raw_shards.items():
    shard_path = RAW_SHARD_PATHS[shard_id]
    shard_df.to_csv(shard_path, index=False)
    if shard_df.empty:
        sample_range = "empty"
        original_preview = "empty"
    else:
        sample_range = f"{int(shard_df['sample_id'].min())}..{int(shard_df['sample_id'].max())}"
        original_preview = f"{shard_df['original_id'].iloc[0]} -> {shard_df['original_id'].iloc[-1]}"
    print(f"Saved raw_shard{shard_id}: {len(shard_df)} rows, sample_id range {sample_range}, original_id range {original_preview}, path {shard_path}")

print("Split midpoint:", midpoint)
print("Total rows after split:", sum(len(frame) for frame in raw_shards.values()))


Primary ordering column: json_file
Original ID source: raw_identifier
Duplicate base original IDs before deterministic suffixing: 0
Unique original_id values: 12644
Saved raw_shard1: 6322 rows, sample_id range 0..6321, original_id range 00049d038a2abc2d5fe3b190d6cf5c1cb1ba63441defdf136be251c7a00727d8 -> 7b83c2d06e5f5bd88739720efb1f280741b383bd164ac1ed4ea5f644e113113b, path E:\PFE_CoT_outputs\data\interim\raw_shard1.csv
Saved raw_shard2: 6322 rows, sample_id range 6322..12643, original_id range 7b85db656d0ecfcef6a32f073955be280c9fd12a7676030c2642a9bf2d4ff28f -> fff8b8b6d87726c902d76b2159801cee2a6d492e2970ce3a63f2651006cb76f0, path E:\PFE_CoT_outputs\data\interim\raw_shard2.csv
Split midpoint: 6322
Total rows after split: 12644


7'-This cell optionally deletes only the unassigned intermediate raw shard while leaving `data/raw/` untouched.

In [8]:
# Keep only the assigned intermediate shard when requested.
# This never deletes anything in data/raw and never deletes ZIP files or extracted raw folders.
opposite_shard_id = 2 if ASSIGNED_SHARD_ID == 1 else 1
opposite_path = RAW_SHARD_PATHS[opposite_shard_id]

if DELETE_UNASSIGNED_SHARD:
    if opposite_path.exists():
        opposite_path.unlink()
        print(f"Kept assigned shard {ASSIGNED_SHARD_ID}: {ASSIGNED_RAW_SHARD_PATH}")
        print(f"Removed unassigned intermediate shard {opposite_shard_id}: {opposite_path}")
    else:
        print(f"Kept assigned shard {ASSIGNED_SHARD_ID}: {ASSIGNED_RAW_SHARD_PATH}")
        print(f"Unassigned intermediate shard {opposite_shard_id} was already absent: {opposite_path}")
else:
    print("DELETE_UNASSIGNED_SHARD is False; keeping both raw shard CSV files.")


DELETE_UNASSIGNED_SHARD is False; keeping both raw shard CSV files.


In [9]:
# Cell added to extract the longest and shorted json files id
saved = pd.read_csv(JSON_SUMMARY_PATH)
small_hash = saved.nsmallest(1, "dynamic_host_event_count")["original_id"].iloc[0]
large_hash = saved.nlargest(1, "dynamic_host_event_count")["original_id"].iloc[0]

small_path = next(Path(JSON_DIR).glob(f"*{small_hash[:20]}*"), None)
large_path = next(Path(JSON_DIR).glob(f"*{large_hash[:20]}*"), None)

print("Small:", small_path)
print("Large:", large_path)

Small: E:\dataset\Capturing-logs\Data_Merged\7BA3D0FCC53F4CFF08D7EF68265208BC698D1D1E467441B37DF7A8677DBC319B.json
Large: E:\dataset\Capturing-logs\Data_Merged\8051668DC4B6671DA7585E30EE059438F425777EFFF22BC3566BAA5615413ADD.json


In [10]:
#check zero events samples
import pandas as pd
df = pd.read_csv(r"E:\PFE_CoT_outputs\data\interim\json_behavior_summary_shard2.csv")
event_count_column = "dynamic_host_event_count"
zero_events = df[df[event_count_column].fillna(0).astype(int) == 0]
display_columns = [
    column for column in ["original_id", "sample_id", "json_sha256", "json_md5", "json_process_name", "json_package"]
    if column in zero_events.columns
]
print(f"Zero-event samples: {len(zero_events)}")
print(zero_events[display_columns].to_string(index=False))

Zero-event samples: 26
                                                     original_id  sample_id                                                      json_sha256 json_md5 json_process_name json_package
7ba3d0fcc53f4cff08d7ef68265208bc698d1d1e467441b37df7a8677dbc319b       6327 7ba3d0fcc53f4cff08d7ef68265208bc698d1d1e467441b37df7a8677dbc319b  unknown           unknown      unknown
7ca99c298ddb5b4e3aca2ef389fd59433bd49af19dc7111fce07a0f3c10d7061       6386 7ca99c298ddb5b4e3aca2ef389fd59433bd49af19dc7111fce07a0f3c10d7061  unknown           unknown      unknown
7cd4ae1d7a195e04fd44d667752ac123708204f7e14c602591ac95c078f2a4ef       6391 7cd4ae1d7a195e04fd44d667752ac123708204f7e14c602591ac95c078f2a4ef  unknown           unknown      unknown
7d05b21cc30b212b6e9814b3bdddf775ae8e410f31bd2d5c59355f39eb05b8c4       6399 7d05b21cc30b212b6e9814b3bdddf775ae8e410f31bd2d5c59355f39eb05b8c4  unknown           unknown      unknown
7d742ba848410c4c6d686d5af556e4b3f940cf28c460bd4a06ebc98753bfbd51       6

8-This cell loads the assigned raw shard, verifies `original_id`, and enriches only that shard with compact JSON behavior summaries from `Data_Merged`.

In [11]:
# From this point onward, inspect and clean only the assigned shard.
if not ASSIGNED_RAW_SHARD_PATH.exists():
    raise FileNotFoundError(f"Assigned shard is missing: {ASSIGNED_RAW_SHARD_PATH}")

assigned_df = pd.read_csv(ASSIGNED_RAW_SHARD_PATH, low_memory=False)
if "original_id" not in assigned_df.columns:
    raise KeyError("Assigned raw shard must contain original_id. Re-run notebook 01 from the raw CICMalDroid files.")
if "json_path" not in assigned_df.columns:
    raise KeyError("Assigned raw shard must contain json_path so dynamic behavior can be summarized.")

print("Assigned raw shard shape before JSON enrichment:", assigned_df.shape)
print("Assigned sample_id range:", int(assigned_df["sample_id"].min()), "to", int(assigned_df["sample_id"].max()))
print("Assigned original_id preview:", assigned_df["original_id"].head(3).tolist())

JSON_SUMMARY_CHECKPOINT_INTERVAL = 50
assigned_original_ids = set(assigned_df["original_id"].astype(str))
json_summary_records_running = []
cached_original_ids = set()

if JSON_SUMMARY_PATH.exists() and not REBUILD_JSON_SUMMARY:
    print("Loading cached JSON behavior summary:", JSON_SUMMARY_PATH)
    cached_summary_df = pd.read_csv(JSON_SUMMARY_PATH, low_memory=False)
    if "original_id" not in cached_summary_df.columns:
        raise KeyError("JSON summary cache must contain original_id.")
    cached_summary_df["original_id"] = cached_summary_df["original_id"].astype(str)
    cached_summary_df = cached_summary_df.drop_duplicates(subset=["original_id"], keep="first")
    stale_cache_mask = ~cached_summary_df["original_id"].isin(assigned_original_ids)
    stale_cache_count = int(stale_cache_mask.sum())
    if stale_cache_count:
        print("Ignoring cached records outside this assigned shard:", stale_cache_count)
    cached_summary_df = cached_summary_df.loc[~stale_cache_mask].reset_index(drop=True)
    json_summary_records_running = cached_summary_df.to_dict("records")
    cached_original_ids = set(cached_summary_df["original_id"].astype(str))
elif JSON_SUMMARY_PATH.exists() and REBUILD_JSON_SUMMARY:
    print("Ignoring existing JSON behavior summary cache because REBUILD_JSON_SUMMARY=True:", JSON_SUMMARY_PATH)
else:
    print("No cached JSON behavior summary found:", JSON_SUMMARY_PATH)

rows_to_process = assigned_df.loc[~assigned_df["original_id"].astype(str).isin(cached_original_ids)].copy()
print("JSON behavior summaries already cached:", len(cached_original_ids))
print("JSON behavior summaries remaining to process:", len(rows_to_process))

if len(rows_to_process):
    print("Building/resuming JSON behavior summary for assigned shard only. This may take time for large JSON files.")
    newly_processed_count = 0
    for _, row in tqdm(rows_to_process.iterrows(), total=len(rows_to_process)):
        json_path = Path(row["json_path"])
        record = summarize_json_behavior(json_path)
        record["original_id"] = str(row["original_id"])
        record["sample_id"] = int(row["sample_id"])
        json_summary_records_running.append(record)
        newly_processed_count += 1
        if newly_processed_count % JSON_SUMMARY_CHECKPOINT_INTERVAL == 0:
            summary_df = pd.DataFrame(json_summary_records_running)
            summary_df.to_csv(JSON_SUMMARY_PATH, index=False)
            print(f"Saved JSON summary checkpoint: {len(json_summary_records_running)}/{len(assigned_df)} records -> {JSON_SUMMARY_PATH}")
else:
    print("JSON behavior summary cache already covers this assigned shard.")

summary_df = pd.DataFrame(json_summary_records_running)
summary_df.to_csv(JSON_SUMMARY_PATH, index=False)
print(f"Saved complete JSON behavior summary cache: {len(summary_df)}/{len(assigned_df)} records -> {JSON_SUMMARY_PATH}")

if "original_id" not in summary_df.columns:
    raise KeyError("JSON summary cache must contain original_id.")

summary_columns_to_drop = [column for column in ["sample_id"] if column in summary_df.columns]
summary_df = summary_df.drop(columns=summary_columns_to_drop)
assigned_df = assigned_df.merge(summary_df, on="original_id", how="left", validate="one_to_one")

json_summary_text = assigned_df["json_behavior_summary"].fillna("").astype(str)
json_loaded_mask = assigned_df["json_loaded"].astype("string").str.strip().str.lower().isin(["true", "1", "yes"]) if "json_loaded" in assigned_df.columns else pd.Series(False, index=assigned_df.index)
invalid_json_mask = json_summary_text.str.startswith("json_parse_error") | ~json_loaded_mask
invalid_json_drop_count = int(invalid_json_mask.sum())
assigned_df = assigned_df.loc[~invalid_json_mask].reset_index(drop=True)
print("Dropped rows with JSON parse errors or unloaded JSON:", invalid_json_drop_count)
print("Rows remaining after JSON validity filter:", len(assigned_df))

print("Assigned shard shape after JSON enrichment:", assigned_df.shape)
print("Assigned raw shard columns:")
print(assigned_df.columns.tolist())
print("JSON files successfully parsed:", int(assigned_df["json_loaded"].fillna(False).astype(bool).sum()) if "json_loaded" in assigned_df.columns else 0)
display(assigned_df.head())

Assigned raw shard shape before JSON enrichment: (6322, 14)
Assigned sample_id range: 6322 to 12643
Assigned original_id preview: ['7b85db656d0ecfcef6a32f073955be280c9fd12a7676030c2642a9bf2d4ff28f', '7b86dd2ea1402d153f151d46acaa184c152074e7349201d0e58e9658be065f3f', '7b8bb0f41cea2f08eb22c0ac092239b822895f926079f1fd7bd562503a6e8a22']
Loading cached JSON behavior summary: E:\PFE_CoT_outputs\data\interim\json_behavior_summary_shard2.csv
JSON behavior summaries already cached: 349
JSON behavior summaries remaining to process: 5973
Building/resuming JSON behavior summary for assigned shard only. This may take time for large JSON files.


  0%|          | 0/5973 [00:00<?, ?it/s]

KeyboardInterrupt: 

8'-Manual emergency save for the JSON behavior summary cache.

In [12]:
# MANUAL EMERGENCY SAVE: run this at any time during an interruption to save the in-memory
# JSON summary records to JSON_SUMMARY_PATH without corrupting the cache.
if "json_summary_records_running" not in globals():
    raise NameError("json_summary_records_running is not available yet. Start the JSON summarization cell first.")
if not json_summary_records_running:
    raise ValueError("No JSON summary records are in memory; refusing to overwrite the cache with an empty file.")

emergency_summary_df = pd.DataFrame(json_summary_records_running)
emergency_summary_df.to_csv(JSON_SUMMARY_PATH, index=False)
print("Manual emergency save records in memory:", len(json_summary_records_running))
print("Manual emergency save written:", JSON_SUMMARY_PATH)


Manual emergency save records in memory: 352
Manual emergency save written: E:\PFE_CoT_outputs\data\interim\json_behavior_summary_shard2.csv


9-This cell inspects missing values, inferred label/category/family columns, class distributions, and duplicate counts for the assigned shard.

In [13]:
missing_summary = (
    assigned_df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_percent=lambda data: (data["missing_count"] / len(assigned_df) * 100).round(2))
    .sort_values(["missing_count", "missing_percent"], ascending=False)
)
print("Missing values by column (top 50):")
display(missing_summary.head(50))

label_col = find_first_column(assigned_df.columns, LABEL_CANDIDATES)
category_col = find_first_column(assigned_df.columns, CATEGORY_CANDIDATES)
family_col = find_first_column(assigned_df.columns, FAMILY_CANDIDATES)
identifier_col = "raw_identifier" if "raw_identifier" in assigned_df.columns else find_first_column(assigned_df.columns, IDENTIFIER_CANDIDATES)

print("Detected identifier column:", identifier_col)
print("Detected label column:", label_col)
print("Detected category column:", category_col)
print("Detected family column:", family_col)

for title, column in [("Label", label_col), ("Category", category_col), ("Family", family_col)]:
    if column and column in assigned_df.columns:
        print(f"\n{title} distribution from column '{column}' (top 30):")
        print(assigned_df[column].value_counts(dropna=False).head(30))
    else:
        print(f"\n{title} distribution: no likely column detected")

print("\nDuplicate checks on assigned shard")
print("Full duplicate rows:", int(assigned_df.duplicated().sum()))
if identifier_col and identifier_col in assigned_df.columns:
    print(f"Duplicate identifiers in '{identifier_col}':", int(assigned_df[identifier_col].duplicated().sum()))


Missing values by column (top 50):


,missing_count,missing_percent
sample_id,0,0.0
original_id,0,0.0
source_dataset,0,0.0
raw_identifier,0,0.0
json_file,0,0.0
json_path,0,0.0
sha256,0,0.0
true_label,0,0.0
category,0,0.0
family,0,0.0


Detected identifier column: raw_identifier
Detected label column: true_label
Detected category column: category
Detected family column: family

Label distribution from column 'true_label' (top 30):
true_label
SMS         1883
Riskware    1530
Benign      1145
Banking     1091
Adware       673
Name: count, dtype: int64

Category distribution from column 'category' (top 30):
category
SMS         1883
Riskware    1530
Benign      1145
Banking     1091
Adware       673
Name: count, dtype: int64

Family distribution from column 'family' (top 30):
family
unknown    6322
Name: count, dtype: int64

Duplicate checks on assigned shard
Full duplicate rows: 0
Duplicate identifiers in 'raw_identifier': 0


10-This cell identifies behavioral evidence columns while excluding metadata, labels, identifiers, and empty columns.

In [14]:
metadata_norms = {normalize_name(value) for value in [
    identifier_col, label_col, category_col, family_col,
    "sample_id", "original_id", "source_dataset", "source_file", "source_row_order",
] if value}
metadata_keywords = [
    "label", "class", "category", "family", "verdict", "hash", "sha", "md5",
    "identifier", "package", "filename", "sourcefile",
]

evidence_columns = []
for column in assigned_df.columns:
    key = normalize_name(column)
    if key in metadata_norms:
        continue
    if any(keyword in key for keyword in metadata_keywords):
        continue
    if assigned_df[column].isna().all():
        continue
    evidence_columns.append(column)

text_evidence_columns = [column for column in evidence_columns if assigned_df[column].dtype == "object"]
numeric_evidence_columns = [column for column in evidence_columns if column not in text_evidence_columns]

print("Evidence columns detected:", len(evidence_columns))
print("Text/object evidence columns:", len(text_evidence_columns))
print("Numeric/boolean evidence columns:", len(numeric_evidence_columns))
print("First 80 evidence columns:")
print(evidence_columns[:80])


Evidence columns detected: 2
Text/object evidence columns: 0
Numeric/boolean evidence columns: 2
First 80 evidence columns:
['json_file', 'json_path']


11-This cell performs light shard-local cleaning, preserves `original_id`, creates canonical metadata fields, and saves the assigned cleaned shard.

In [15]:
# Light cleaning on the assigned shard only. Rows are preserved to keep shard ownership stable.
cleaned_df = assigned_df.copy()
if "original_id" not in cleaned_df.columns:
    raise KeyError("original_id must never be dropped. Re-run notebook 01 from the raw CICMalDroid files.")

empty_columns = [column for column in cleaned_df.columns if cleaned_df[column].isna().all() and column != "original_id"]
if empty_columns:
    print("Dropping fully empty columns from assigned shard:", empty_columns)
    cleaned_df = cleaned_df.drop(columns=empty_columns)
else:
    print("No fully empty columns to drop.")


def value_or_unknown(series):
    if series is None:
        return pd.Series(["unknown"] * len(cleaned_df), index=cleaned_df.index, dtype="string")
    return series.astype("string").fillna("unknown").replace("", "unknown")


label_col = find_first_column(cleaned_df.columns, LABEL_CANDIDATES)
category_col = find_first_column(cleaned_df.columns, CATEGORY_CANDIDATES)
family_col = find_first_column(cleaned_df.columns, FAMILY_CANDIDATES)
identifier_col = "raw_identifier" if "raw_identifier" in cleaned_df.columns else find_first_column(cleaned_df.columns, IDENTIFIER_CANDIDATES)
true_label_source = label_col or category_col or family_col

if "true_label" not in cleaned_df.columns:
    cleaned_df["true_label"] = value_or_unknown(cleaned_df[true_label_source]) if true_label_source else "unknown"
if "category" not in cleaned_df.columns:
    cleaned_df["category"] = value_or_unknown(cleaned_df[category_col]) if category_col else cleaned_df["true_label"]
if "family" not in cleaned_df.columns:
    cleaned_df["family"] = value_or_unknown(cleaned_df[family_col]) if family_col else "unknown"
if identifier_col and identifier_col in cleaned_df.columns and "raw_identifier" not in cleaned_df.columns:
    cleaned_df["raw_identifier"] = value_or_unknown(cleaned_df[identifier_col])
elif "raw_identifier" not in cleaned_df.columns:
    cleaned_df["raw_identifier"] = "unknown"

cleaned_df.to_csv(CLEANED_PATH, index=False)
print("Saved assigned cleaned CICMalDroid shard:", CLEANED_PATH)
print("Rows preserved:", len(cleaned_df))
print("Unique original_id values:", cleaned_df["original_id"].nunique())
print("\nFinal true_label distribution (top 30):")
print(cleaned_df["true_label"].value_counts(dropna=False).head(30))
display(cleaned_df.head())


No fully empty columns to drop.
Saved assigned cleaned CICMalDroid shard: E:\PFE_CoT_outputs\data\interim\cleaned_cicmaldroid_shard2.csv
Rows preserved: 6322
Unique original_id values: 6322

Final true_label distribution (top 30):
true_label
SMS         1883
Riskware    1530
Benign      1145
Banking     1091
Adware       673
Name: count, dtype: int64


,sample_id,original_id,source_dataset,raw_identifier,json_file,json_path,sha256,true_label,category,family,label_source_file,label_conflict,source_file,source_row_order
0,6322,7b85db656d0ecfcef6a32f073955be280c9fd12a767603...,CICMalDroid2020,7b85db656d0ecfcef6a32f073955be280c9fd12a767603...,7B85DB656D0ECFCEF6A32F073955BE280C9FD12A767603...,E:\dataset\Capturing-logs\Data_Merged\7B85DB65...,7b85db656d0ecfcef6a32f073955be280c9fd12a767603...,SMS,SMS,unknown,sms.txt,False,E:\dataset\Capturing-logs\Data_Merged\7B85DB65...,6322
1,6323,7b86dd2ea1402d153f151d46acaa184c152074e7349201...,CICMalDroid2020,7b86dd2ea1402d153f151d46acaa184c152074e7349201...,7B86DD2EA1402D153F151D46ACAA184C152074E7349201...,E:\dataset\Capturing-logs\Data_Merged\7B86DD2E...,7b86dd2ea1402d153f151d46acaa184c152074e7349201...,Benign,Benign,unknown,benign.txt,False,E:\dataset\Capturing-logs\Data_Merged\7B86DD2E...,6323
2,6324,7b8bb0f41cea2f08eb22c0ac092239b822895f926079f1...,CICMalDroid2020,7b8bb0f41cea2f08eb22c0ac092239b822895f926079f1...,7B8BB0F41CEA2F08EB22C0AC092239B822895F926079F1...,E:\dataset\Capturing-logs\Data_Merged\7B8BB0F4...,7b8bb0f41cea2f08eb22c0ac092239b822895f926079f1...,Banking,Banking,unknown,banking.txt,False,E:\dataset\Capturing-logs\Data_Merged\7B8BB0F4...,6324
3,6325,7b8f075b62149ab08af06589bb75f7dda3668ebb1d127e...,CICMalDroid2020,7b8f075b62149ab08af06589bb75f7dda3668ebb1d127e...,7B8F075B62149AB08AF06589BB75F7DDA3668EBB1D127E...,E:\dataset\Capturing-logs\Data_Merged\7B8F075B...,7b8f075b62149ab08af06589bb75f7dda3668ebb1d127e...,Riskware,Riskware,unknown,riskware.txt,False,E:\dataset\Capturing-logs\Data_Merged\7B8F075B...,6325
4,6326,7ba3a5c0d7ae0dc7da4d0a5343b8a03d006f68f26908ac...,CICMalDroid2020,7ba3a5c0d7ae0dc7da4d0a5343b8a03d006f68f26908ac...,7BA3A5C0D7AE0DC7DA4D0A5343B8A03D006F68F26908AC...,E:\dataset\Capturing-logs\Data_Merged\7BA3A5C0...,7ba3a5c0d7ae0dc7da4d0a5343b8a03d006f68f26908ac...,SMS,SMS,unknown,sms.txt,False,E:\dataset\Capturing-logs\Data_Merged\7BA3A5C0...,6326
